# 02. Análisis de mercado

**Pasos:**

- Evolución anual de primas emitidas

- Comisiones directas: montos y porcentajes

- Primas retenidas: montos y porcentajes

- Distribución anual de primas por tipo de embaración

- Monto anual total de siniestros 

- Distribución anual de siniestros por causa

- Conclusión general del análisis del mercado 

In [28]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display


# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar nuestras funciones
sys.path.insert(0, str(ROOT))

from funciones_reservas import leer_snapshot

# Definir las carpetas de trabajo
PROCESSED = ROOT / "data" / "processed"


# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------

# Recuperar las bases previamente ajustadas a diciembre de 2025
emision_ajustada = leer_snapshot(
    PROCESSED / "emision_ajustada_2025.parquet"
)

siniestros_ajustados = leer_snapshot(
    PROCESSED / "siniestros_ajustados_2025.parquet"
)

# Mostrar las dimensiones de las bases
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")


Emisión: (21353, 11)
Siniestros: (2763, 14)


## 2.1. Análisis de la prima emitida


### 2.1.1 Comparación de la evolución del mercado

- ¿Cómo ha evulucionado el mercado de seguros de embarcaciones?
- ¿qué proporción de ellas se retiene o destina a comisiones?

In [29]:
# --------------------------------------------------
# 2.1.1 ANÁLISIS DE LA PRIMA EMITIDA
# --------------------------------------------------

# 1. Agrupar los montos ajustados por año
columnas_monetarias = [
    "prima_emitida_ajustada",
    "prima_retenida_ajustada",
    "comision_directa_ajustada"
]

emision_anual = (
    emision_ajustada
    .groupby("año")[columnas_monetarias]
    .sum()
    .sort_index()
)

# 2. Calcular los indicadores anuales

# Tasas de crecimiento de las primas (%)
columnas_primas = [
    "prima_emitida_ajustada",
    "prima_retenida_ajustada"
]

emision_anual[["crecimiento_emitida", "crecimiento_retenida"]] = (
    emision_anual[columnas_primas].pct_change() * 100
)

# Proporción de comisiones respecto a la prima emitida (%)
emision_anual["%Com/Prima"] = (
    emision_anual["comision_directa_ajustada"]
    / emision_anual["prima_emitida_ajustada"] * 100
)

# 3. Normalizar las variables (año base 2015 = 100)
columnas_comparacion = columnas_primas + ["%Com/Prima"]

comparacion = emision_anual[columnas_comparacion].copy()

# Todas las series comienzan en 100 para comparar su evolución
comparacion = (
    comparacion
    .div(comparacion.loc[2015])
    .mul(100)
    .reset_index()
)

# 4. Visualizar la evolución relativa
fig = px.line(
    comparacion,
    x="año",
    y=columnas_comparacion,
    markers=True,
    title="Evolución relativa de primas y comisiones ",
    labels={
        "año": "Año",
        "value": "Índice ",
        "variable": "Indicador"
    }
)


fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        x=0.03,
        y=0.95,
        font=dict(size=20)
    ),

    # Espacios
    margin=dict(
        l=65,
        r=40,
        b=65,
        t=130
    ),

    # Interacción
    hovermode="x unified",
    showlegend=True,

    # Dimensiones
    height=550
)

# Eje X
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False
)

# Eje Y
fig.update_yaxes(
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=False,
    showline=False
)

fig.show()

Con fines comparativos, se normalizaron las variables tomando 2015 como año base (índice de 100), para observar sus cambios relativos y facilitar la comparación de los comportamientos a lo largo de los años.

Se observa una tendencia creciente en las primas emitidas, alcanzando un aumento aproximado del 150% en 2025 respecto a 2015 lo cual el aumento de este último año puede deberse a los conflictos entre Estados Unidos e Irán. También destaca una caída importante en 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. 

Por otro lado, las primas retenidas se mantienen relativamente estables a pesar del crecimiento de las emitidas, esto podría deberse a una mayor participación del reaseguro, lo que implicaría una menor proporción de riesgos asumidos directamente por las aseguradoras. 

Finalmente, la proporción de comisiones aumentó durante 2020 y 2021, posiblemente porque las primas emitidas disminuyeron más que las comisiones, mientras que en los últimos años esta proporción se redujo, lo que sugiere un menor peso relativo de las comisiones dentro del mercado.

### 2.1.2 Distribución de las primas de embarcación



- ¿Cuáles aportan más?
- ¿Como cambian las participaciones con el tiempo?

In [30]:
# --------------------------------------------------
#  2.1.2 Distribución de prima por tipo de embarcación
# --------------------------------------------------

import plotly.express as px

# 1. Agrupar las primas por año y tipo de embarcación
participacion = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion"])["prima_emitida_ajustada"]
    .sum()
    .unstack(fill_value=0)
)


# 2. Seleccionar los 6 tipos con mayor prima acumulada
principales = (
    participacion.sum()
    .nlargest(6)
    .index
)

# 3. Agrupar los demás tipos en una sola categoría
participacion["Otros"] = (
    participacion.drop(columns=principales).sum(axis=1)
)

participacion = participacion[
    list(principales) + ["Otros"]
]

# 4. Calcular la participación porcentual anual
participacion = (
    participacion
    .div(participacion.sum(axis=1), axis=0)
    .mul(100)
)

# 5. Convertir a formato largo
participacion = (
    participacion
    .reset_index()
    .melt(
        id_vars="año",
        var_name="Tipo de embarcación",
        value_name="Participación (%)"
    )
)

# 6. Graficar
fig = px.area(
    participacion,
    x="año",
    y="Participación (%)",
    color="Tipo de embarcación",
    title="Distribución anual de primas por tipo de embarcación"
)


fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        x=0.03,
        y=0.95,
        font=dict(size=20)
    ),

    # Espacios
    margin=dict(
        l=65,
        r=40,
        b=65,
        t=130
    ),

    # Interacción
    hovermode="x unified",
    showlegend=True,

    # Dimensiones
    height=550
)

# Eje X
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False
)

# Eje Y
fig.update_yaxes(
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=False,
    showline=False
)

fig.show()

Con el objetivo de analizar cómo ha cambiado la composición del mercado, se calculó la participación porcentual de cada tipo de embarcación respecto a la prima emitida total de cada año.

Se observa que entre 2015 y 2022 los abastecedores tuvieron una participación importante, destacando especialmente en 2019, sin embargo, a partir de 2023 los buques tanque petroleros comenzaron a dominar el mercado, alcanzando aproximadamente el 77% de las primas emitidas en 2025. Este cambio podría estar relacionado con lo antes ya mensionado de las tensiónes entre Estados Unidos e Irán, con ello aumento de los riesgos en el transporte marítimo de petróleo, que han provocado incrementos en los costos de aseguramiento de ciertas rutas, lo que podría explicar una mayor concentración de primas en este tipo de embarcaciones.

Esto sugiere que el mercado se ha vuelto más dependiente del segmento petrolero, lo que podría representar una mayor exposición a los riesgos asociados con esta actividad y una menor diversificación de las primas emitidas.

## 2.2 Análisis del monto de siniestros

### 2.2.1 Distribución del monto de siniestros por causa y cuanto cambia por año

- ¿Cuándo ocurren las mayores perdidas?
- ¿Cuáles son las principales causas de perdida?


In [31]:
# --------------------------------------------------
# 2.2.1.  Distribución anual de pérdidas por causa
# --------------------------------------------------

import plotly.express as px

# 1. Identificar las columnas monetarias ajustadas
columnas_pagos = [
    f"{i}_monto pagado_ajustada" for i in range(4)
]

# 2. Calcular el monto total de cada registro
siniestros_analisis = siniestros_ajustados.copy()

siniestros_analisis["monto_total"] = (
    siniestros_analisis[columnas_pagos]
    .sum(axis=1, min_count=1)
)

# 3. Agrupar los siniestros por año y causa
siniestros_anual = (
    siniestros_analisis
    .groupby(["año", "causa_siniestro"], as_index=False)["monto_total"]
    .sum()
    .sort_values("año")
)

# 4. Construir la gráfica de barras apiladas
fig = px.bar(
    siniestros_anual,
    x="año",
    y="monto_total",
    color="causa_siniestro",
    barmode="relative",
    title="Distribución anual de siniestros por causa",
    labels={
        "año": "Año de origen",
        "monto_total": "Monto de siniestros ajustado (MXN)",
        "causa_siniestro": "Causa del siniestro"
    }
)


fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        x=0.03,
        y=0.95,
        font=dict(size=20)
    ),

    # Espacios
    margin=dict(
        l=65,
        r=40,
        b=65,
        t=130
    ),

    # Interacción
    hovermode="x unified",
    showlegend=True,

    # Dimensiones
    height=550
)

# Eje X
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False
)

# Eje Y
fig.update_yaxes(
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=False,
    showline=False
)

fig.show()

Se observa que 2015 presenta el mayor monto de siniestros del periodo, con aproximadamente 4,200 millones de pesos, de los cuales al rededor de 3,800 millones son principalmente en fenómenos naturales, esto podría relacionarse con la intensa temporada de huracanes registrada ese año, especialmente con el paso del huracán Patricia, lo que refleja la exposición del mercado marítimo a eventos naturales capaces de generar pérdidas importantes. Posteriormente, los montos disminuyen, aunque vuelven a presentar incrementos en 2023 y 2024, con una mayor participación de distintas causas, como varaduras y encallamientos, lo que muestra que las pérdidas del mercado también dependen de riesgos operativos y no únicamente de fenómenos naturales.

## 2.3 Conclusión del Análisis de Mercado 

Se observa que el mercado de seguros de embarcaciones presenta un crecimiento general en las primas emitidas, aunque con fluctuaciones importantes, como la caída de 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. Por otro lado, las primas se han concentrado cada vez más en los buques tanque petroleros, posiblemente debido a una mayor actividad o al incremento de los costos de aseguramiento en este sector por conflictos geopolíticos, lo que implica una mayor dependencia del mercado hacia este tipo de embarcaciones.

En cuanto a los siniestros, destaca 2015 por las elevadas pérdidas asociadas a fenómenos naturales, posiblemente relacionadas con la temporada de huracanes, especialemente el paso del huracán Patricia, mientras que en años posteriores se observa una mayor participación de causas operativas, como varaduras y encallamientos. 

En conjunto, los resultados muestran un mercado en crecimiento, pero con una importante concentración de primas y exposición a pérdidas extraordinarias, lo que resalta la importancia de una adecuada diversificación y gestión de riesgos.